In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

c:\Users\Prayush\OneDrive\Desktop\Sem V\Advanced Python for DS\fedspeak analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FinBERT sentiment analysis

In [2]:
statements = pd.read_csv('../data/raw/fomc_statements.csv')
minutes = pd.read_csv('../data/raw/fomc_minutes.csv')
speeches = pd.read_csv('../data/raw/fed_speeches.csv')

In [3]:
# Load FinBERT model
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

print("FinBERT loaded!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12148.47it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded!


In [4]:
def get_finbert_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', 
                      max_length=512, 
                      truncation=True, 
                      padding=True)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Convert to probabilities
    probs = torch.softmax(outputs.logits, dim=1).squeeze()  #basically pytorch's tensor of probabilities
    
    # FinBERT output order: positive, negative, neutral
    return {
        'positive': probs[0].item(),
        'negative': probs[1].item(),
        'neutral': probs[2].item()
    }
test = "The Federal Reserve will raise interest rates to combat elevated inflation."
print(get_finbert_sentiment(test))

{'positive': 0.7921574115753174, 'negative': 0.025361904874444008, 'neutral': 0.18248066306114197}


text
 ↓
tokenizer(text)
 ↓
model(inputs)
 ↓
logits
 ↓
softmax
 ↓
sentiment probabilities

In [5]:
import spacy
nlp = spacy.load('en_core_web_sm')

def score_document(text):
    if not isinstance(text, str):
        return None
    
    # Split into sentences
    doc = nlp(text) #tokenization + lemmatization
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    
    if not sentences:
        return None
    
    # Score each sentence
    pos_scores, neg_scores, neu_scores = [], [], []
    
    for sentence in sentences:
        scores = get_finbert_sentiment(sentence)
        pos_scores.append(scores['positive'])
        neg_scores.append(scores['negative'])
        neu_scores.append(scores['neutral'])
    
    return {
        'positive': np.mean(pos_scores),
        'negative': np.mean(neg_scores),
        'neutral': np.mean(neu_scores)
    }

# Test on first statement
sample = statements['text'][0]
print(score_document(sample))

{'positive': np.float64(0.32864700433694655), 'negative': np.float64(0.3177778826923006), 'neutral': np.float64(0.35357511162550914)}


In [35]:
#applies 'score_document' to every statement, minute and speech to generate finbert_scores
print("Scoring statements...")
statements['finbert_scores'] = statements['text'].apply(score_document)

print("Scoring minutes...")
minutes['finbert_scores'] = minutes['text'].apply(score_document)

print("Scoring speeches...")
speeches['finbert_scores'] = speeches['text'].apply(score_document)

print("Done!")

for df in [statements, minutes, speeches]:
    df['positive'] = df['finbert_scores'].apply(lambda x: x['positive'] if x else None)
    df['negative'] = df['finbert_scores'].apply(lambda x: x['negative'] if x else None)
    df['neutral'] = df['finbert_scores'].apply(lambda x: x['neutral'] if x else None)
    df.drop(columns=['finbert_scores'], inplace=True)

Scoring statements...
Scoring minutes...
Scoring speeches...
Done!


In [7]:
#REMOVE THIS AFTER CSV IS CREATED
statements.to_csv('../data/finbertscores/statements_finbert.csv', index=False)
minutes.to_csv('../data/finbertscores/minutes_finbert.csv', index=False)
speeches.to_csv('../data/finbertscores/speeches_finbert.csv', index=False)
print("All saved!")

All saved!


In [8]:
statements_finbert= pd.read_csv('../data/finbertscores/statements_finbert.csv')
minutes_finbert= pd.read_csv('../data/finbertscores/minutes_finbert.csv')
speeches_finbert=pd.read_csv('../data/finbertscores/speeches_finbert.csv')

statements_finbert.head()

,date,meeting,url,text
0,2010-01-27,"January 27, 2010",https://www.federalreserve.gov/newsevents/pres...,Information received since the Federal Open Ma...
1,2010-03-16,"March 16, 2010",https://www.federalreserve.gov/newsevents/pres...,Information received since the Federal Open Ma...
2,2010-04-28,"April 28, 2010",https://www.federalreserve.gov/newsevents/pres...,Information received since the Federal Open Ma...
3,2010-05-09,"May 9, 2010",https://www.federalreserve.gov/newsevents/pres...,In response to the reemergence of strains in U...
4,2010-06-23,"June 23, 2010",https://www.federalreserve.gov/newsevents/pres...,Information received since the Federal Open Ma...


In [9]:
print(statements.columns)

Index(['date', 'meeting', 'url', 'text'], dtype='str')


Keyword scoring

In [10]:
# Hawkish and dovish keyword lists
hawkish_keywords = [
    'restrictive', 'tightening', 'tighten', 'elevated', 'pressure', 
    'headwinds', 'inflation', 'rate hike', 'firm', 'aggressive', 
    'restraint', 'robust', 'strong', 'necessity', 'urgency', 
    'warranted', 'appropriate', 'frontloaded', 'above target'
]

dovish_keywords = [
    'accommodative', 'support', 'downside', 'slack', 'uncertainty', 
    'patient', 'gradual', 'transitory', 'flexible', 'temporary', 
    'manageable', 'moderate', 'sustained', 'watchful', 'cautious', 
    'careful', 'cushion', 'buffer', 'insurance'
]

def keyword_score(text):
    if not isinstance(text, str):
        return 0
    
    text = text.lower()
    total_tokens = len(text.split())
    
    hawk_count = sum(text.count(word) for word in hawkish_keywords)
    dove_count = sum(text.count(word) for word in dovish_keywords)
    
    hawk_freq = hawk_count / (total_tokens + 1e-6)
    dove_freq = dove_count / (total_tokens + 1e-6)
    
    # Normalized score: positive = hawkish, negative = dovish
    score = (hawk_freq - dove_freq) / (hawk_freq + dove_freq + 1e-6)
    return score

# Test
print(keyword_score("The Federal Reserve will raise interest rates to combat elevated inflation."))
print(keyword_score("The committee remains patient and accommodative to support the gradual recovery."))

0.9999945000297499
-0.9999972500073125


In [11]:
statements_processed = pd.read_csv('../data/preprocessed/fomc_statements_processed.csv')
minutes_processed = pd.read_csv('../data/preprocessed/fomc_minutes_processed.csv')
speeches_processed = pd.read_csv('../data/preprocessed/fed_speeches_processed.csv')

In [12]:
statements_processed['keyword_score'] = statements_processed['processed_text'].apply(keyword_score)
minutes_processed['keyword_score'] = minutes_processed['processed_text'].apply(keyword_score)
speeches_processed['keyword_score'] = speeches_processed['processed_text'].apply(keyword_score)

In [ ]:
statements_processed.to_csv('../data/preprocessed/fomc_statements_processed.csv', index=False)
minutes_processed.to_csv('../data/preprocessed/fomc_minutes_processed.csv', index=False)
speeches_processed.to_csv('../data/preprocessed/fed_speeches_processed.csv', index=False)